# AudioStory — Audio-to-Video Pipeline

This notebook takes a 30-second audio clip, analyzes it deeply, generates an AI-written storyboard, and produces a beat-synced video edit with consistent characters and scenes.

**Pipeline stages:**
1. Audio Analysis (librosa + Whisper)
2. Storyboard Generation (Groq LLM)
3. Image Generation (HuggingFace Inference API)
4. Video Effects + Assembly (OpenCV + MoviePy)

⏱️ Total runtime: ~2-4 minutes on a free Colab T4 GPU.

## Cell 1: Install Dependencies

In [ ]:
%pip install -q librosa openai-whisper groq huggingface_hub Pillow opencv-python moviepy numpy scipy python-dotenv
print('Dependencies installed ✓')

## Cell 2: Mount Google Drive

Mount Google Drive to save output videos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/AudioStory_Outputs'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

## Cell 3: Set API Keys

Enter your Groq API key and HuggingFace token below.
- Groq: https://console.groq.com (free tier)
- HuggingFace: https://huggingface.co/settings/tokens (free)

In [ ]:
import os

GROQ_API_KEY = 'your_groq_api_key_here'
HF_TOKEN = 'your_huggingface_token_here'

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['OUTPUT_DIR'] = OUTPUT_DIR

from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)

from huggingface_hub import InferenceClient
hf_client = InferenceClient(token=HF_TOKEN)

print('API clients initialized ✓')

## Cell 4: Upload Audio File

Upload a 30-second audio clip (mp3, wav, m4a, or ogg).

In [ ]:
from google.colab import files
import uuid

uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
audio_path = f'/content/{audio_filename}'

print(f'Audio file: {audio_filename}')
print(f'File size: {os.path.getsize(audio_path) / 1024:.1f} KB')

## Cell 5: Stage 1 — Audio Analysis

Extract tempo, beats, onsets, energy, spectral features, mood, and transcribe lyrics using librosa + Whisper.

In [ ]:
import librosa
import numpy as np
import whisper
import json

print('[Stage 1] Loading and analyzing audio...')

# Load audio
y, sr = librosa.load(audio_path, duration=30, mono=True)
duration = float(librosa.get_duration(y=y, sr=sr))
print(f'  Duration: {duration:.2f}s, Sample rate: {sr}Hz')

# Tempo and beats
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, units='frames')
tempo = float(tempo) if hasattr(tempo, '__iter__') else float(tempo)
beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
print(f'  Tempo: {tempo:.1f} BPM, Beats: {len(beat_times)}')

# Onset detection
onset_times = librosa.onset.onset_detect(
    y=y, sr=sr, units='time', delta=0.2, wait=5
).tolist()
print(f'  Onsets: {len(onset_times)}')

# RMS energy
hop_length = 512
rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]
rms_max = rms.max() if rms.max() > 0 else 1.0
rms_normalized = (rms / rms_max).tolist()
rms_timestamps = librosa.frames_to_time(
    np.arange(len(rms)), sr=sr, hop_length=hop_length
).tolist()

# Spectral centroid (brightness)
sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
sc_max = sc.max() if sc.max() > 0 else 1.0
sc_normalized = (sc / sc_max)
sc_mean = float(np.mean(sc_normalized))

# Chroma
chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
chroma_mean = [float(v) for v in np.mean(chroma, axis=1)]

# MFCCs
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mfcc_mean = [float(v) for v in np.mean(mfcc, axis=1)]

# Zero crossing rate
zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))

# Mood detection
avg_energy = float(np.mean(rms_normalized))

def detect_mood(energy, brightness, tempo, zcr):
    if energy > 0.7 and tempo > 130:
        return 'intense_aggressive'
    elif energy > 0.7 and tempo > 100:
        return 'energetic_euphoric'
    elif energy > 0.5 and brightness > 0.6:
        return 'uplifting_heroic'
    elif energy > 0.5 and brightness < 0.4:
        return 'dark_powerful'
    elif energy < 0.3 and brightness < 0.4:
        return 'melancholic_lonely'
    elif energy < 0.3 and brightness > 0.6:
        return 'peaceful_dreamy'
    elif zcr > 0.1:
        return 'chaotic_anxious'
    else:
        return 'neutral_cinematic'

detected_mood = detect_mood(avg_energy, sc_mean, tempo, zcr)
print(f'  Mood: {detected_mood}')

# Energy sections
n = len(rms_normalized)
section_size = max(1, n // 4)
labels = ['intro', 'buildup', 'climax', 'outro']
energy_sections = []
for i, label in enumerate(labels):
    start_idx = i * section_size
    end_idx = start_idx + section_size if i < 3 else n
    if start_idx >= n: break
    section_rms = rms_normalized[start_idx:end_idx]
    avg = float(np.mean(section_rms)) if section_rms else 0.0
    energy_sections.append({
        'label': label,
        'start': float(rms_timestamps[start_idx]),
        'end': float(rms_timestamps[min(end_idx - 1, len(rms_timestamps) - 1)]),
        'avg_energy': avg,
        'energy_label': 'high' if avg > 0.6 else 'medium' if avg > 0.3 else 'low'
    })

# Transcription
print('  Loading Whisper... (may take a moment)')
whisper_model = whisper.load_model('base')
result = whisper_model.transcribe(audio_path)
transcript = result['text'].strip()
print(f'  Transcript: {transcript[:80]}')

# Build result dict
audio_data = {
    'duration': duration,
    'tempo': tempo,
    'beat_times': beat_times,
    'onset_times': onset_times,
    'rms_values': rms_normalized,
    'rms_timestamps': rms_timestamps,
    'spectral_centroid_mean': sc_mean,
    'spectral_centroid_over_time': sc_normalized.tolist(),
    'chroma_mean': chroma_mean,
    'mfcc_mean': mfcc_mean,
    'zero_crossing_rate': zcr,
    'transcript': transcript,
    'detected_mood': detected_mood,
    'energy_sections': energy_sections
}

# Save analysis
analysis_path = f'{OUTPUT_DIR}/audio_analysis_{audio_filename}.json'
with open(analysis_path, 'w') as f:
    json.dump(audio_data, f, indent=2)

# Display waveform and beat visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(rms_timestamps, rms_normalized, color='cyan', alpha=0.7)
axes[0].set_title('RMS Energy Over Time')
axes[0].set_ylabel('Energy (normalized)')
axes[0].axhline(y=0.6, color='r', linestyle='--', alpha=0.5)
axes[0].axhline(y=0.3, color='orange', linestyle='--', alpha=0.5)
for on in onset_times:
    axes[0].axvline(x=on, color='red', alpha=0.3)

times = np.linspace(0, duration, len(rms_normalized))
axes[1].plot(times, rms_normalized, color='cyan')
for bt in beat_times:
    axes[1].axvline(x=bt, color='yellow', alpha=0.3, linestyle='--')
axes[1].set_title('Beats (yellow dashed)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Energy')

plt.tight_layout()
plt.show()

print(f'[Stage 1] Complete. Analysis saved to: {analysis_path}')

## Cell 6: Choose Visual Style

In [ ]:
styles = {
    'mha_anime': {
        'display_name': 'MHA Anime',
        'description': 'Shonen anime style inspired by My Hero Academia',
        'trigger_word': 'mha style, anime, shonen, vibrant colors, hero costume, dynamic pose'
    },
    'cyberpunk': {
        'display_name': 'Cyberpunk Neon',
        'description': 'Dark neon-lit cyberpunk aesthetic',
        'trigger_word': 'cyberpunk style, neon lights, dark city, futuristic, holograms, rain'
    },
    'lofi_pastel': {
        'display_name': 'Lofi Pastel',
        'description': 'Soft pastel tones, lofi anime illustration',
        'trigger_word': 'lofi style, soft pastel, warm illustration, gentle, cozy, dreamy'
    },
    'dark_fantasy': {
        'display_name': 'Dark Fantasy',
        'description': 'Epic dark fantasy illustration style',
        'trigger_word': 'dark fantasy art, epic illustration, cinematic, dramatic, medieval, armor'
    },
    'minimalist': {
        'display_name': 'Minimalist',
        'description': 'Clean flat minimalist illustration',
        'trigger_word': 'minimalist illustration, flat design, clean lines, simple, geometric, limited colors'
    }
}

style_key = 'cyberpunk'  # Change this to your preferred style
style = styles[style_key]
print(f'Selected style: {style["display_name"]}')
print(f'Trigger words: {style["trigger_word"]}')

## Cell 7: Stage 2 — Storyboard Generation (Groq)

Send audio features to Groq's LLM and receive a structured JSON storyboard.

In [ ]:
print('[Stage 2] Generating storyboard via Groq...')

# Determine scene count
high_energy = sum(1 for s in energy_sections if s['energy_label'] == 'high')
if high_energy >= 3:
    num_scenes = 5
elif high_energy >= 2:
    num_scenes = 4
else:
    num_scenes = 3

# Build prompt
from utils.prompt_builder import build_storyboard_prompt
import sys
sys.path.insert(0, '/content')

# Reconstruct a simple audio data object for the prompt builder
class SimpleAudioData:
    def __init__(self, d):
        for k, v in d.items():
            setattr(self, k, v)

audio_obj = SimpleAudioData(audio_data)

prompt = f"""
You are creating a storyboard for a 30-second music video edit.

AUDIO ANALYSIS DATA
Duration: {audio_data['duration']:.1f} seconds
Tempo: {audio_data['tempo']:.1f} BPM
Mood: {audio_data['detected_mood']}
Brightness: {'bright and energetic' if audio_data['spectral_centroid_mean'] > 0.5 else 'dark and heavy'}
Lyrics: {audio_data['transcript'] or 'instrumental'}

Energy Sections: {json.dumps(audio_data['energy_sections'], indent=2)}

Style: {style['display_name']} — {style['description']}
Style trigger: {style['trigger_word']}

Generate a storyboard JSON with exactly {num_scenes} scenes.
Each scene has keyframes (full image) and continuations (img2img modification).
The same character must appear in every scene.
Use this JSON format:
{{
  "main_story": "...",
  "subject_description": "...",
  "subject_reference_prompt": "...",
  "total_scenes": {num_scenes},
  "scenes": [
    {{
      "scene_id": 1,
      "start_time": 0.0,
      "end_time": 8.0,
      "narrative": "...",
      "setting": "...",
      "mood": "...",
      "color_palette": "...",
      "transition_from_previous": "cut",
      "frames": [
        {{
          "frame_id": 1,
          "timestamp": 0.0,
          "type": "KEYFRAME",
          "full_prompt": "extremely detailed prompt with character description, pose, setting, lighting, style, quality tags",
          "subject_pose": "standing, looking forward",
          "camera_angle": "medium shot",
          "lighting": "dramatic lighting"
        }},
        {{
          "frame_id": 2,
          "timestamp": 0.5,
          "type": "CONTINUATION",
          "modifier_prompt": "slight head turn, camera pan left",
          "strength": 0.2,
          "what_changes": "character turns head slightly"
        }}
      ]
    }}
  ]
}}
Respond with ONLY valid JSON. Do not add any text before or after."
]

response = groq_client.chat.completions.create(
    model='llama3-70b-8192',
    messages=[{'role': 'user', 'content': prompt}],
    temperature=0.8,
    max_tokens=4096
)

raw = response.choices[0].message.content.strip()
# Strip code fences if present
import re
raw = re.sub(r'^```json\s*', '', raw, flags=re.MULTILINE)
raw = re.sub(r'\s*```$', '', raw, flags=re.MULTILINE)

storyboard = json.loads(raw)

# Save
storyboard_path = f'{OUTPUT_DIR}/storyboard_{audio_filename}.json'
with open(storyboard_path, 'w') as f:
    json.dump(storyboard, f, indent=2)

print(f'[Stage 2] Storyboard complete!')
print(f'  Main story: {storyboard["main_story"][:100]}...')
print(f'  Scenes: {storyboard["total_scenes"]}')
for scene in storyboard['scenes']:
    n_kf = sum(1 for f in scene['frames'] if f['type'] == 'KEYFRAME')
    n_cont = sum(1 for f in scene['frames'] if f['type'] == 'CONTINUATION')
    print(f'  Scene {scene["scene_id"]}: {n_kf} keyframes, {n_cont} continuations — {scene["narrative"][:60]}')

## Cell 8: Stage 3 — Image Generation (HuggingFace)

Generate the reference image and walk through every frame.
- KEYFRAME: fresh img2img from reference (strength=0.82)
- CONTINUATION: img2img from previous frame (strength=0.15-0.40)

In [ ]:
from PIL import Image

neg_prompt = (
    'bad anatomy, deformed, ugly, blurry, low quality, watermark, '
    'text, signature, extra limbs, mutated, distorted face, '
    'inconsistent character, different person, wrong style, nsfw'
)

frames_dir = f'{OUTPUT_DIR}/frames_{audio_filename}'
os.makedirs(frames_dir, exist_ok=True)

# ── Generate reference image ──────────────────────────────────────
print('[Stage 3] Generating reference image...')
ref_prompt = f"{storyboard['subject_reference_prompt']}, {style['trigger_word']}, masterpiece, best quality"

ref_image = hf_client.text_to_image(
    prompt=ref_prompt,
    negative_prompt=neg_prompt,
    width=512, height=512,
    num_inference_steps=30,
    guidance_scale=7.5
)
ref_image.save(os.path.join(frames_dir, 'reference.png'))
print(f'  Reference saved ✓')

# ── Generate all frames ───────────────────────────────────────────
all_frames = []
for scene in storyboard['scenes']:
    for frame in scene['frames']:
        all_frames.append(frame)

prev_image = ref_image.copy()
total = len(all_frames)

for i, frame in enumerate(all_frames):
    fpath = os.path.join(frames_dir, f'frame_{frame["frame_id"]:03d}.png')
    print(f'  [{i+1}/{total}] {frame["type"]} @ t={frame["timestamp"]:.2f}s')

    if frame['type'] == 'KEYFRAME':
        prompt = f"{frame['full_prompt']}, {style['trigger_word']}, {scene['color_palette']}, masterpiece, best quality, detailed"
        img = hf_client.image_to_image(
            image=ref_image,
            prompt=prompt,
            strength=0.82,
            negative_prompt=neg_prompt,
            num_inference_steps=30,
            guidance_scale=7.5
        )
    else:
        prompt = f"{storyboard['subject_description']}, {frame['modifier_prompt']}, {scene['color_palette']}, {style['trigger_word']}"
        img = hf_client.image_to_image(
            image=prev_image,
            prompt=prompt,
            strength=frame.get('strength', 0.22),
            negative_prompt=neg_prompt,
            num_inference_steps=30,
            guidance_scale=7.5
        )

    img.save(fpath)
    prev_image = img.copy()
    time.sleep(0.5)  # rate limit

# Display results
from google.colab import display
print(f'[Stage 3] All {total} frames generated ✓')
print(f'Frames directory: {frames_dir}')
# Show first few frames
for frame in all_frames[:4]:
    img = Image.open(os.path.join(frames_dir, f'frame_{frame["frame_id"]:03d}.png'))
    display.display(img.resize((256, 256)))

## Cell 9: Stage 4 — Video Assembly with Beat-Synced Effects

Render each frame with visual effects driven by the audio data, then
compose into final video with original audio track.

In [ ]:
import cv2
from moviepy import ImageSequenceClip, AudioFileClip, CompositeVideoClip

TARGET_FPS = 30
OUTPUT_SIZE = (1080, 1080)

def apply_ken_burns(image, t, duration, energy):
    zoom_speed = 0.04 + (energy * 0.06)
    progress = t / max(duration, 0.001)
    zoom = 1.0 + (zoom_speed * progress)
    h, w = image.shape[:2]
    new_w, new_h = int(w * zoom), int(h * zoom)
    resized = cv2.resize(image, (new_w, new_h))
    x_off, y_off = (new_w - w) // 2, (new_h - h) // 2
    return resized[y_off:y_off+h, x_off:x_off+w]

def apply_flash(image, strength):
    white = np.ones_like(image, dtype=np.uint8) * 255
    return cv2.addWeighted(image, 1.0 - strength, white, strength, 0)

def apply_glitch(image, intensity):
    img = image.copy()
    shift = max(1, int(intensity * 25))
    img[:, shift:, 2] = image[:, :-shift, 2]
    img[:, :-shift, 0] = image[:, shift:, 0]
    n_slices = int(intensity * 8) + 2
    for _ in range(n_slices):
        y = np.random.randint(0, max(1, img.shape[0] - 20))
        sh = np.random.randint(2, min(18, img.shape[0] - y))
        ss = np.random.randint(-35, 35)
        img[y:y+sh] = np.roll(img[y:y+sh], ss, axis=1)
    return img

def apply_color_grade(image, energy, brightness):
    img = image.astype(np.float32)
    if brightness > 0.55:
        img[:,:,2] = np.clip(img[:,:,2] * (1.0 + energy*0.35), 0, 255)
        img[:,:,0] = np.clip(img[:,:,0] * (1.0 - energy*0.15), 0, 255)
    else:
        img[:,:,0] = np.clip(img[:,:,0] * (1.0 + energy*0.35), 0, 255)
        img[:,:,2] = np.clip(img[:,:,2] * (1.0 - energy*0.15), 0, 255)
    if energy > 0.6:
        hsv = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,1] = np.clip(hsv[:,:,1] * (1.0 + energy*0.4), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32)
    return np.clip(img, 0, 255).astype(np.uint8)

def apply_vignette(image, strength):
    h, w = image.shape[:2]
    Y, X = np.ogrid[:h, :w]
    cx, cy = w/2, h/2
    dist = np.sqrt(((X-cx)/cx)**2 + ((Y-cy)/cy)**2)
    mask = np.clip(1.0 - dist*strength, 0, 1)[:, :, np.newaxis]
    return (image * mask).astype(np.uint8)

def apply_zoom(image, zoom_factor):
    h, w = image.shape[:2]
    new_w, new_h = int(w*zoom_factor), int(h*zoom_factor)
    resized = cv2.resize(image, (new_w, new_h))
    x_off, y_off = (new_w-w)//2, (new_h-h)//2
    return resized[y_off:y_off+h, x_off:x_off+w]

def get_energy_at(t):
    idx = int(np.searchsorted(audio_data['rms_timestamps'], t))
    idx = min(idx, len(audio_data['rms_values']) - 1)
    return float(audio_data['rms_values'][idx])

def get_brightness_at(t):
    idx = int(np.searchsorted(audio_data['rms_timestamps'], t))
    idx = min(idx, len(audio_data['spectral_centroid_over_time']) - 1)
    return float(audio_data['spectral_centroid_over_time'][idx])

def is_near_onset(onsets, t, window=0.08):
    return any(abs(t - o) <= window for o in onsets)

print('[Stage 4] Rendering frames with effects...')

# Build frame list with calculated durations
frame_list = storyboard['scenes']
rendered_frames = []
all_clips = []

for scene_idx, scene in enumerate(storyboard['scenes']):
    scene_frames = scene['frames']
    for f_idx, frame in enumerate(scene_frames):
        # Determine frame duration
        if f_idx + 1 < len(scene_frames):
            duration = scene_frames[f_idx+1]['timestamp'] - frame['timestamp']
        else:
            duration = scene['end_time'] - frame['timestamp']
        duration = max(0.01, duration)

        # Load and resize
        img = cv2.imread(os.path.join(frames_dir, f'frame_{frame["frame_id"]:03d}.png'))
        img = cv2.resize(img, OUTPUT_SIZE)

        num_video_frames = max(1, int(duration * TARGET_FPS))
        energy = get_energy_at(frame['timestamp'])
        brightness = get_brightness_at(frame['timestamp'])
        is_onset = is_near_onset(audio_data['onset_times'], frame['timestamp'])

        print(f'  Frame {frame["frame_id"]}: {duration:.2f}s, {num_video_frames} vframes, energy={energy:.2f}')

        frame_imgs = []
        for vf in range(num_video_frames):
            t = vf / TARGET_FPS
            current = img.copy()

            # Ken Burns (always)
            current = apply_ken_burns(current, t, duration, energy)

            # Beat flash on keyframes
            if vf < 3 and frame['type'] == 'KEYFRAME':
                current = apply_flash(current, min(0.7, energy * 0.9))

            # Glitch on onsets
            if is_onset and vf < 5:
                current = apply_glitch(current, intensity=energy * 0.6)

            # Color grade (always)
            current = apply_color_grade(current, energy, brightness)

            # Vignette at low energy
            if energy < 0.35:
                current = apply_vignette(current, strength=0.6 - energy)

            # Zoom punch at high energy onsets
            if is_onset and energy > 0.8 and vf < 8:
                zoom = 1.0 + (0.06 * (1.0 - vf / 8))
                current = apply_zoom(current, zoom)

            current = cv2.cvtColor(current, cv2.COLOR_BGR2RGB)
            frame_imgs.append(current)

        clip = ImageSequenceClip(frame_imgs, fps=TARGET_FPS)
        clip = clip.with_start(frame['timestamp'])
        all_clips.append(clip)

print('[Stage 4] Compositing final video...')
final = CompositeVideoClip(all_clips, size=OUTPUT_SIZE)

# Add audio
audio_clip = AudioFileClip(audio_path).subclip(0, min(30, audio_data['duration']))
final = final.with_audio(audio_clip)

# Export
output_path = f'{OUTPUT_DIR}/audiostory_{audio_filename.replace(".mp3","").replace(".wav","")}.mp4'
final.write_videofile(output_path, fps=TARGET_FPS, codec='libx264', audio_codec='aac')

print(f'[Stage 4] Video saved to: {output_path}')

## Cell 10: Preview and Download

In [ ]:
from google.colab import files
from IPython.display import Video

print(f'Output: {output_path}')
Video(output_path, width=400, height=400, embed=True)

files.download(output_path)
print('✓ Download ready')

## Summary

This notebook walks through the complete AudioStory pipeline:
1. **Audio Analysis** — Extract tempo, beats, onsets, energy, mood, and transcribe lyrics
2. **Storyboard Generation** — Structured JSON plan via Groq LLM with consistent characters
3. **Image Generation** — Reference image + keyframe/continuation frame generation via HF API
4. **Video Assembly** — Beat-synced effects (flash, glitch, zoom, color grade, vignette) composited into final MP4

**Tips:**
- Use `base` Whisper model for fastest transcription, `small` for higher accuracy
- Lower `strength` values (0.15-0.22) for subtle continuations, higher for dramatic scenes
- The same character description is included in every prompt for consistency
- Output videos save to Google Drive for persistence